<a href="https://colab.research.google.com/github/pymood1/GPT-SoVITS-improve/blob/main/Colab-Inference-Fast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS Improve — Fast Full Install

기능은 줄이지 않고 설치/다운로드만 최적화한 테스트 노트북입니다. **Colab Runtime Version 2026.07 + GPU** 권장.

앞으로 설치 로직은 `tools/colab_fast_setup.py`만 갱신하므로 이 노트북 링크는 계속 그대로 사용할 수 있습니다.


## Google Drive 마운트

설치 전에 Google Drive를 먼저 연결합니다.


In [ ]:
from google.colab import drive


drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import time


REPO_URL = "https://github.com/pymood1/GPT-SoVITS-improve.git"
REPO_DIR = Path("/content/GPT-SoVITS")


print("[1/2] GPT-SoVITS-improve 최신 main 준비", flush=True)

start = time.perf_counter()


if (REPO_DIR / ".git").exists():
    print("[git] 기존 repo 유지 + 최신 main 동기화", flush=True)

    subprocess.run(
        ["git", "fetch", "--depth", "1", "origin", "main"],
        cwd=REPO_DIR,
        check=True,
    )

    subprocess.run(
        ["git", "reset", "--hard", "origin/main"],
        cwd=REPO_DIR,
        check=True,
    )

else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    print("[git] 새 repo shallow clone", flush=True)

    clone_cmd = [
        "git",
        "clone",
        "--progress",
        "--depth",
        "1",
        "--single-branch",
        "--branch",
        "main",
        REPO_URL,
        str(REPO_DIR),
    ]

    subprocess.run(clone_cmd, check=True)


elapsed = time.perf_counter() - start
print(f"[1/2] repo 준비 완료: {elapsed:.1f}s", flush=True)


print("[2/2] 전체 의존성 + pretrained 설치 시작", flush=True)

setup_script = REPO_DIR / "tools" / "colab_fast_setup.py"


print("[check] setup script 문법 검사", flush=True)

subprocess.run(
    [sys.executable, "-m", "py_compile", str(setup_script)],
    check=True,
)


print("[run] setup script 실행", flush=True)

process = subprocess.Popen(
    [sys.executable, "-u", str(setup_script)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"setup script failed with exit code {return_code}. "
        "위에 표시된 마지막 traceback/에러 메시지를 확인하세요."
    )


## V5dev 파인튜닝 WebUI — 한국어

설치가 끝난 뒤 실행합니다. 처음 열리는 화면은 **V5dev가 선택된 1B-미세조정**이며, UI 언어는 **한국어(ko_KR)** 로 고정합니다.

추론은 같은 WebUI의 **1C-추론**에서 열 수 있고, 생성된 공개 링크가 해당 화면에 표시됩니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


REPO_DIR = Path("/content/GPT-SoVITS")


env = os.environ.copy()

python_path = [
    str(REPO_DIR),
    str(REPO_DIR / "GPT_SoVITS"),
]

existing_python_path = env.get("PYTHONPATH")

if existing_python_path:
    python_path.append(existing_python_path)


env.update(
    {
        "PYTHONPATH": os.pathsep.join(python_path),
        "language": "ko_KR",
        "version": "v5dev",
        "is_share": "True",
    }
)


webui_cmd = [
    sys.executable,
    "-u",
    "webui.py",
    "ko_KR",
]


print("[run] 파인튜닝 WebUI 시작 — 한국어", flush=True)

process = subprocess.Popen(
    webui_cmd,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"webui.py failed with exit code {return_code}. "
        "위에 표시된 마지막 traceback/에러 메시지를 확인하세요."
    )
